# **Digital Twin for Production System**

To simulate a production system in a single environment like Google Colab, we create a "Digital Twin" of a real-world architecture. In this simulation, we will run the Server (the AI model) and the Client (the User/App) in the same notebook but as separate logical processes.

This allows your students to see the Request-Response Lifecycle in real-time.

## **The Production Simulation (Google Colab Version)**

**Step 1: Install the "Waiter" (FastAPI)**

Run this to install the web framework and the background runner.

In [ ]:
!pip install fastapi uvicorn nest-asyncio


**Step 2: The Simulation Core**

run this entire block. It defines the model, starts the API in the background, and then "pings" it like a real user would.

In [ ]:
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import nest_asyncio
import threading
import requests
import time

# --- 1. THE PRODUCTION SERVER (FASTAPI) ---
app = FastAPI()

# Simulated Trained Weights from our Linear Regression Lab
W = np.array([[0.15], [-2.5], [-10.0]]) # [SqFt, Age, Dist]
B = 150.0
MU = np.array([2650.0, 20.0, 8.0])
SIGMA = np.array([1000.0, 11.0, 4.0])

class PredictionRequest(BaseModel):
    sqft: float
    age: float
    distance: float

@app.post("/predict")
async def predict_price(item: PredictionRequest):
    # Scale inputs to match training distribution
    input_data = np.array([[item.sqft, item.age, item.distance]])
    scaled_x = (input_data - MU) / SIGMA

    # Mathematical Inference: y = Xw + b
    raw_prediction = (scaled_x @ W + B)[0][0]

    return {
        "status": "success",
        "prediction_k": f"${round(float(raw_prediction), 2)}k",
        "processed_by": "Multivariate-LR-Engine-v1"
    }

# --- 2. THE BACKGROUND RUNNER ---
# This starts the server in a separate thread so the notebook stays interactive
def run_server():
    nest_asyncio.apply()
    uvicorn.run(app, host="127.0.0.1", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2) # Give the server 2 seconds to "wake up"
print("🚀 PRODUCTION SERVER IS ONLINE AT http://127.0.0.1:8000")

INFO:     Started server process [38674]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


🚀 PRODUCTION SERVER IS ONLINE AT http://127.0.0.1:8000


**Step 3: The "Client" Simulation**

Now, run this cell to act as the user. It will send different scenarios to the API and print the "Log" of what happened.

In [ ]:
# Simulated Users (Different House Scenarios)
scenarios = [
    {"sqft": 3500, "age": 5, "distance": 2, "label": "Luxury Central"},
    {"sqft": 1200, "age": 35, "distance": 12, "label": "Old Suburban"},
    {"sqft": 2200, "age": 15, "distance": 6, "label": "Standard Family"}
]

print("📡 CLIENT: Sending requests to Production API...\n")

for s in scenarios:
    # 1. Construct the JSON 'Envelope'
    payload = {k: v for k, v in s.items() if k != 'label'}

    # 2. POST the data to the API
    response = requests.post("http://127.0.0.1:8000/predict", json=payload)

    # 3. Handle the returned data
    if response.status_code == 200:
        res_data = response.json()
        print(f"🏠 HOUSE TYPE: {s['label']}")
        print(f"   Input: {payload}")
        print(f"   Output: {res_data['prediction_k']}")
        print(f"   Engine: {res_data['processed_by']}\n")
    else:
        print(f"❌ Error: {response.status_code}")

print("✅ Simulation Complete.")

📡 CLIENT: Sending requests to Production API...

INFO:     127.0.0.1:54308 - "POST /predict HTTP/1.1" 200 OK
🏠 HOUSE TYPE: Luxury Central
   Input: {'sqft': 3500, 'age': 5, 'distance': 2}
   Output: $168.54k
   Engine: Multivariate-LR-Engine-v1

INFO:     127.0.0.1:54314 - "POST /predict HTTP/1.1" 200 OK
🏠 HOUSE TYPE: Old Suburban
   Input: {'sqft': 1200, 'age': 35, 'distance': 12}
   Output: $136.37k
   Engine: Multivariate-LR-Engine-v1

INFO:     127.0.0.1:54318 - "POST /predict HTTP/1.1" 200 OK
🏠 HOUSE TYPE: Standard Family
   Input: {'sqft': 2200, 'age': 15, 'distance': 6}
   Output: $156.07k
   Engine: Multivariate-LR-Engine-v1

✅ Simulation Complete.


# **Detailed Breakdown for Students**

1. **The Separation of Concerns**: Even though everything is in one Colab notebook, the Server and Client are physically separate. The Server (Thread 1) is sitting and waiting, while the Client (Thread 2) is sending HTTP messages. This is exactly how Amazon or Uber works.

2. **The JSON Protocol:** Notice that the Server doesn't receive a "Python Object"; it receives a JSON string.Serialization: The Client turns data into a string.Deserialization: The Server turns the string back into data.This allows a model written in Python to talk to an App written in JavaScript or Swift.

3. **The "State" of the Model:** In production, we don't have the original dataset. We only have the Constants ($W$ and $B$).


**Question:** If we lose the MU and SIGMA (the scaling values), can the production model still work?

Answer: No. Without the scaling factors, the input numbers ($3500$) would be 100x larger than what the model expects, and the prediction would be astronomical.

4. **Logging and Status:** A production system always returns a Status Code (like 200 for OK or 404 for Not Found). Our simulation shows how we can include metadata, like which version of the engine processed the request.